## Définition du champ d'analyse des résultats des élections municipales

Ce notebook consiste à définir un périmètre d'analyse permettant de limiter la part des listes sans étiquette politique, particulièrement nombreuses dans la base de données préparée.

L'objectif est ainsi de constituer un sous-champ d'analyse dans lequel les candidatures disposant d'une nuance politique sont davantage représentées, pour ne pas biaiser l'analyse. 

Pour ce faire, nous cherchons à identifier un critère de sélection objectif, indépendant des résultats électoraux et des nuances politiques observées. Examiner les liens entre la population des communes et la présence d'une étiquette politique constitue une première piste.

Cette étape vise donc à déterminer un périmètre de sélection pertinent, offrant un compromis entre la limitation des candidatures sans étiquette et la conservation d'un volume suffisant de données pour la suite de l'étude.

(écrit par IA, remanié manuellement)

In [ ]:
#Imports utiles 
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs
import sys
sys.path.append('../outils')

import fonctions_nomenclature as fnom
import fonctions_preparation_table as fprep
%load_ext autoreload
%autoreload 2

np.random.seed(123)

# Charger la table depuis le bucket 
# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"


FILE_KEY_S3 = "table_analyse_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df = pd.read_csv(file_in, sep=",")  

In [ ]:
# Constat 

tab = df['ind1'].value_counts(dropna=False).rename_axis('Modalité de ind1').reset_index(name='Effectif')

tab['Modalité de ind1'] = tab['Modalité de ind1'].replace({
    2: 'Sans étiquette',
    1: 'Droite',
    0: 'Gauche'
})

tab = tab.style.hide(axis='index')

tab

In [ ]:
# Discrétiser la taille de la commune 

df['POPULATION_dis'] = (pd.cut(df['POPULATION'], np.array([0,3500,10000,max(df['POPULATION'])]), 
                            labels = ["Moins de 3 500 habitants", "Entre 3 500 et 10 000 habitants", "10 000 habitants et plus"],
                            right=False)
)

tab = df['POPULATION_dis'].value_counts()
tab 


In [ ]:
# Statistique bivariée taille commune et ind1
pd.crosstab(df['ind1'], df['POPULATION_dis'])

In [ ]:
# Discrétiser le taux de pauvreté  

df['TX_PAUV_dis'] = (pd.cut(df['TAUX_PAUVRETE'], np.array([0,5,10,15,20,25,30,max(df['TAUX_PAUVRETE'].dropna())]), 
                            labels = ["Moins de 5 %", "Entre 5 et 10 %", "Entre 10 et 15 %", "Entre 15 et 20 %", "Entre 20 et 25 %", "Entre 25 % et 30 %", "30 % et plus"],
                            right=False)
)

tab = df['TX_PAUV_dis'].value_counts()
tab 


In [ ]:
# Statistique bivariée taille commune et ind1
pd.crosstab(df['ind1'], df['TX_PAUV_dis']) 

In [ ]:
# Discrétiser le niveau de vie médian 

df['NIVVIE_MEDIAN_dis'] = (pd.cut(df['NIVVIE_MEDIAN'], np.array([0,10000,15000,20000,25000,30000,40000,max(df['NIVVIE_MEDIAN'].dropna())]), 
                            labels = ["Moins de 10 000", "Entre 10 000 et 15 000", "Entre 15 000 et 20 000", "Entre 20 000 et 25 000", "Entre 25 000 et 30 000", "Entre 30 000 et 40 000", "40 000 et plus"],
                            right=False)
)

tab = df['NIVVIE_MEDIAN_dis'].value_counts()
tab 

In [ ]:
#Code amélioré par IA pour avoir des tranches équivalentes et des labels automatiques 

# Calcul des quantiles et récupération des bornes
_, bornes = pd.qcut(
    df['NIVVIE_MEDIAN'],
    q=5,
    retbins=True,
    duplicates='drop'
)

# Création automatique des labels
labels = [
    f"{bornes[i]:,.0f} € à {bornes[i+1]:,.0f} €".replace(',', ' ')
    for i in range(len(bornes)-1)
]

# Création des classes
df['NIVVIE_MEDIAN_dis'] = pd.qcut(
    df['NIVVIE_MEDIAN'],
    q=5,
    labels=labels,
    duplicates='drop'
)

tab = df['NIVVIE_MEDIAN_dis'].value_counts()
tab 

In [ ]:
# Statistique bivariée taille commune et ind1
pd.crosstab(df['ind1'], df['NIVVIE_MEDIAN_dis']) 

In [ ]:
# On sélectionne notre sous échantillon

df_sous_ech = df[df['POPULATION'] > 3500]

df_sous_ech.shape[0]

In [ ]:
# On enregistre la table dans le bucket 

FILE_KEY_OUT_S3 = "table_analyse_muni_sous_ech.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    df_sous_ech.to_csv(file_out)

# Pour résumer

Le critère de la taille de la population semble le plus pertinent pour sélectionner un sous-échantillon de commune dont les listes candidates des municipales ont une étiquette. 

En sélectionnant des villes de 3500 habitants et plus, on se retrouve sur un sous-échantillon de communes ayant uniquement des listes candidates avec étiquette aux élections. 

Les villes sans étiquettes sont plutot des villes avec un taux de pauvreté inférieur à la moyenne nationale (15 %). En les enlevant, on réalise l'étude sur un sous échantillon où la pauvreté intense est moins marquée.

Les villes sans étiquettes sont réparties équitablement entre différents niveaux de vie. 

Des statistiques descriptives plus fines sont disponibles dans un notebook du dossier analyse.